# 00. 気候モデル入門ハンズオン：簡易 EBM とプログラミングの基本  
# 00. Hands-on introduction to climate modeling: Simple EBM and programming basics

**対象 / Target**  
大学生〜大学院生向け。  
For undergraduate and graduate students.

---

## この Notebook の位置づけ / Role of this notebook

この Notebook は、後続の海洋 1-box、2-box、3-box モデルへ入る前の導入です。  
This notebook is the introduction before moving to the one-box, two-box, and three-box ocean models.

ここでは、最も単純な気候モデルである **0 次元エネルギーバランスモデル** (0D Energy Balance Model; 0D EBM) を使います。  
Here, we use the simplest climate model, the **zero-dimensional Energy Balance Model** (0D EBM).

この Notebook の目的は、Python を専門的に学ぶことではありません。  
The goal is not to study Python as a programming language in depth.

目的は、次の対応を理解することです。  
The goal is to understand the following correspondence:

```text
物理の考え方
↓
数式
↓
1ステップ更新
↓
for 文による時間積分
↓
図
↓
感度実験
```

```text
physical idea
↓
equation
↓
one-step update
↓
time integration with a for-loop
↓
plot
↓
sensitivity experiment
```

後の海洋ボックスモデルでも、やることは基本的に同じです。  
In the later ocean box models, the basic structure is the same.

## 1. 気候モデルとは何か / What is a climate model?

気候モデルは、地球の気候システムを数式で表し、コンピュータで時間発展を計算する道具です。  
A climate model is a tool that represents the Earth's climate system using equations and computes its time evolution with a computer.

モデルは、現実をそのままコピーするものではありません。  
A model is not a perfect copy of reality.

重要だと考える過程を選び、単純化し、数式にします。  
We choose processes that we think are important, simplify them, and write them as equations.

```text
現実の地球
  ↓ 単純化
概念モデル
  ↓ 数式化
数学モデル
  ↓ プログラム化
Python code
  ↓ 実行
結果・図・考察
```

```text
Real Earth
  ↓ simplification
Conceptual model
  ↓ equations
Mathematical model
  ↓ programming
Python code
  ↓ execution
Results, figures, interpretation
```

気候モデルには、さまざまな階層があります。  
There are many levels of climate models.

| モデル / Model | 特徴 / Feature | 用途 / Use |
|---|---|---|
| 概念モデル / Conceptual model | 変数が少ない / few variables | 本質的な仕組みを理解する / understand essential mechanisms |
| EBM | エネルギー収支を扱う / treats energy balance | 温暖化・気候感度の理解 / understand warming and climate sensitivity |
| ボックスモデル / Box model | 領域を箱に分ける / divides the system into boxes | 炭素循環・物質循環の理解 / understand carbon and tracer cycles |
| GCM | 大気・海洋の流体運動を解く / solves atmospheric and oceanic fluid motion | 気候予測・実験 / climate projections and experiments |
| ESM | 生物地球化学も含む / includes biogeochemistry | 炭素循環・気候フィードバック / carbon cycle and climate feedbacks |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["figure.figsize"] = (7, 4)

## 2. 0 次元 EBM の基本式 / Basic equation of the 0D EBM

全球平均気温偏差を \(T\) とします。  
Let \(T\) be the global-mean temperature anomaly.

0 次元 EBM は次の式で書けます。  
The zero-dimensional EBM can be written as:

\[
C \frac{dT}{dt} = F - \lambda T
\]

ここで、  
where:

- \(T\): 全球平均気温偏差 [K] / global-mean temperature anomaly [K]
- \(C\): 有効熱容量 [W yr m\(^{-2}\) K\(^{-1}\)] / effective heat capacity
- \(F\): 放射強制力 [W m\(^{-2}\)] / radiative forcing
- \(\lambda\): 気候フィードバックパラメタ [W m\(^{-2}\) K\(^{-1}\)] / climate feedback parameter

右辺の \(F\) は気温を上げる効果、\(\lambda T\) は温暖化に伴って宇宙へ逃げるエネルギーの増加を表します。  
On the right-hand side, \(F\) warms the system, while \(\lambda T\) represents the increased energy loss to space as temperature rises.

## 3. まずパラメタを入れる / Define parameters first

いきなり関数を書くのではなく、まず 1 回だけ計算してみます。  
Before writing a function, we first compute one step explicitly.

```text
値を入れる
↓
1ステップ進める
↓
更新する
↓
繰り返す
```

```text
set values
↓
advance one step
↓
update
↓
repeat
```

In [ ]:
F = 3.7       # radiative forcing [W/m2]
lam = 1.2     # climate feedback parameter [W/m2/K]
C = 10.0      # effective heat capacity [W yr/m2/K]

T = 0.0       # initial temperature anomaly [K]

print("F      =", F)
print("lambda =", lam)
print("C      =", C)
print("Initial T =", T)

## 4. 1 ステップだけ進める / Advance only one step

\[
\frac{dT}{dt} = \frac{F - \lambda T}{C}
\]

時間刻みを \(\Delta t\) とすると、前進オイラー法では、

\[
T_{n+1} = T_n + \frac{F - \lambda T_n}{C}\Delta t
\]

と書けます。  
With time step \(\Delta t\), the forward Euler method gives:

\[
T_{n+1} = T_n + \frac{F - \lambda T_n}{C}\Delta t
\]

Python では、これは次のように書けます。  
In Python, this can be written as follows.

In [ ]:
dt = 0.1  # time step [yr]

dTdt = (F - lam * T) / C
T_new = T + dTdt * dt

print("Old T =", T)
print("dTdt  =", dTdt)
print("New T =", T_new)

### Mini exercise 1 / ミニ演習 1

`dt` を 0.1, 1.0, 5.0 に変えると、1 ステップ後の `T_new` はどう変わるでしょうか。  
Change `dt` to 0.1, 1.0, and 5.0. How does `T_new` change after one step?

時間刻みが大きいほど、1 回の更新量は大きくなります。  
A larger time step gives a larger one-step update.

In [ ]:
for dt_test in [0.1, 1.0, 5.0]:
    dTdt = (F - lam * T) / C
    T_new = T + dTdt * dt_test
    print(f"dt = {dt_test:4.1f} yr  T_new = {T_new:.3f} K")

## 5. 更新とは何か / What is an update?

```python
T_new = T + ...
```

は「次の値」を計算しているだけです。  
This only computes the next value.

まだ `T` は変わっていません。  
`T` has not changed yet.

現在値を変えるには、次のように代入します。  
To change the current value, assign the new value back:

```python
T = T_new
```

これは海洋ボックスモデルでも同じです。  
This is the same in ocean box models.

```python
PO4S_new = ...
PO4S = PO4S_new
```

In [ ]:
T = 0.0
dt = 0.1

for n in range(5):
    dTdt = (F - lam * T) / C
    T_new = T + dTdt * dt

    print(f"step {n}: old T = {T:.4f}, new T = {T_new:.4f}")

    # update
    T = T_new

## 6. for 文は時間積分である / A for-loop is time integration

同じ更新を何度も繰り返すと、時間発展になります。  
Repeating the same update many times gives time evolution.

```text
initial value
↓
after 1 step
↓
after 2 steps
↓
after 3 steps
↓
...
```

海洋モデルでも、毎日または毎時間、このような更新を繰り返します。  
Ocean models also repeat this kind of update every day or every time step.

In [ ]:
years = 150
dt = 0.1

time = np.arange(0, years + dt, dt)
T_series = np.zeros_like(time)

T = 0.0

for i in range(len(time)):
    T_series[i] = T
    dTdt = (F - lam * T) / C
    T_new = T + dTdt * dt
    T = T_new

plt.plot(time, T_series)
plt.xlabel("Time [yr]")
plt.ylabel("Temperature anomaly [K]")
plt.title("Zero-dimensional EBM")
plt.grid(True)
plt.show()

print("Final T =", T_series[-1])
print("Equilibrium value F/lambda =", F / lam)

## 7. EBM を関数にする / Put the EBM into a function

同じ計算を何度も使うため、関数にします。  
To reuse the same calculation, we put it into a function.

```text
入力
↓
同じ計算
↓
出力
```

```text
input
↓
same calculation
↓
output
```

In [ ]:
def run_ebm(F=3.7, lam=1.2, C=10.0, years=150, dt=0.1, T0=0.0):
    time = np.arange(0, years + dt, dt)
    T = np.zeros_like(time)
    T[0] = T0

    for n in range(len(time) - 1):
        dTdt = (F - lam * T[n]) / C
        T[n + 1] = T[n] + dTdt * dt

    return time, T

## 8. CO2 倍増を模したステップ強制 / Step forcing for CO2 doubling

大気中二酸化炭素濃度が倍増したときの放射強制力は、よく

\[
F_{2 \times CO2} \approx 3.7\ \mathrm{W\ m^{-2}}
\]

と近似されます。  
The radiative forcing for a doubling of atmospheric CO2 is often approximated as:

\[
F_{2 \times CO2} \approx 3.7\ \mathrm{W\ m^{-2}}
\]

ここでは \(F = 3.7\ \mathrm{W\ m^{-2}}\) の一定強制を与えます。  
Here, we apply a constant forcing of \(F = 3.7\ \mathrm{W\ m^{-2}}\).

In [ ]:
time, T = run_ebm(F=3.7, lam=1.2, C=10.0, years=150)

plt.figure()
plt.plot(time, T)
plt.xlabel("Time [years]")
plt.ylabel("Global mean temperature anomaly [K]")
plt.title("0D EBM: response to CO2 doubling forcing")
plt.grid(True)
plt.show()

## 9. 平衡気候感度 ECS / Equilibrium Climate Sensitivity

平衡状態では、

\[
\frac{dT}{dt}=0
\]

です。  
At equilibrium,

\[
\frac{dT}{dt}=0
\]

したがって、

\[
T_{\mathrm{eq}} = \frac{F}{\lambda}
\]

が得られます。  
Thus,

\[
T_{\mathrm{eq}} = \frac{F}{\lambda}
\]

CO2 倍増強制力に対する平衡温度上昇を **ECS** と呼びます。  
The equilibrium warming for CO2 doubling forcing is called **ECS**.

In [ ]:
F2x = 3.7
lam = 1.2

ECS = F2x / lam

print(f"ECS = {ECS:.2f} K")

## 10. Challenge A: フィードバックを変える / Change climate feedback

\(\lambda\) が大きいほど、温暖化を打ち消す効果が強くなります。  
A larger \(\lambda\) means stronger damping of warming.

**予想 / Prediction**

\(\lambda\) を小さくすると、最終的な温暖化量は大きくなるでしょうか、小さくなるでしょうか。  
If \(\lambda\) becomes smaller, does the final warming become larger or smaller?

In [ ]:
F = 3.7
C = 10.0
lambda_list = [0.7, 1.0, 1.2, 1.5, 2.0]

plt.figure()

for lam in lambda_list:
    time, T = run_ebm(F=F, lam=lam, C=C, years=150)
    ecs = F / lam
    plt.plot(time, T, label=f"lambda={lam}, ECS={ecs:.1f} K")

plt.xlabel("Time [years]")
plt.ylabel("Global mean temperature anomaly [K]")
plt.title("Sensitivity to climate feedback parameter")
plt.legend()
plt.grid(True)
plt.show()

### 問い 1 / Question 1

\(\lambda\) が小さいモデルほど、ECS はどう変化しますか。  
How does ECS change when \(\lambda\) becomes smaller?

また、温度の時間発展はどう変わりますか。  
How does the time evolution of temperature change?

\[
ECS = \frac{F_{2 \times CO2}}{\lambda}
\]

なので、\(\lambda\) が小さいほど ECS は大きくなります。  
Since \(ECS = F_{2 \times CO2}/\lambda\), smaller \(\lambda\) gives larger ECS.

## 11. Challenge B: 熱容量を変える / Change heat capacity

\(C\) は有効熱容量です。  
\(C\) is the effective heat capacity.

海洋の熱吸収が大きいほど、気温上昇はゆっくりになります。  
If ocean heat uptake is larger, warming proceeds more slowly.

**予想 / Prediction**

\(C\) を大きくすると、最終的な温度は変わるでしょうか。  
If \(C\) is increased, does the final equilibrium temperature change?

それとも、平衡に近づく速さだけが変わるでしょうか。  
Or does only the speed of approach to equilibrium change?

In [ ]:
F = 3.7
lam = 1.2
C_list = [5, 10, 30, 100]

plt.figure()

for C in C_list:
    time, T = run_ebm(F=F, lam=lam, C=C, years=300)
    plt.plot(time, T, label=f"C={C}")

plt.xlabel("Time [years]")
plt.ylabel("Global mean temperature anomaly [K]")
plt.title("Sensitivity to heat capacity")
plt.legend()
plt.grid(True)
plt.show()

### 問い 2 / Question 2

\(C\) が大きいモデルでは、温度応答は速くなりますか、遅くなりますか。  
In a model with larger \(C\), does the temperature response become faster or slower?

平衡温度は \(F/\lambda\) で決まり、\(C\) には依存しません。  
The equilibrium temperature is determined by \(F/\lambda\), and does not depend on \(C\).

しかし、平衡へ近づく速さは \(C\) に依存します。  
However, the speed of approach to equilibrium depends on \(C\).

## 12. CO2 が徐々に増える実験 / Gradual CO2 increase experiment

CMIP では、CO2 が毎年 1 % ずつ増える理想化実験がよく使われます。  
In CMIP, an idealized experiment with CO2 increasing by 1% per year is often used.

CO2 が 1 %/yr で増えると、約 70 年で倍増します。  
If CO2 increases by 1% per year, it doubles in about 70 years.

ここでは、放射強制力を時間とともに線形に増えるものとして近似します。  
Here, we approximate radiative forcing as increasing linearly with time.

\[
F(t) = F_{2 \times CO2} \frac{t}{70}
\]

In [ ]:
def forcing_1pctCO2(time, F2x=3.7):
    return F2x * time / 70.0


def run_ebm_timevarying(F_func, lam=1.2, C=10.0, years=150, dt=0.1, T0=0.0):
    time = np.arange(0, years + dt, dt)
    T = np.zeros_like(time)
    F = F_func(time)
    T[0] = T0

    for n in range(len(time) - 1):
        dTdt = (F[n] - lam * T[n]) / C
        T[n + 1] = T[n] + dTdt * dt

    return time, T, F

In [ ]:
time, T, F = run_ebm_timevarying(forcing_1pctCO2, lam=1.2, C=10.0, years=150)

plt.figure()
plt.plot(time, F)
plt.xlabel("Time [years]")
plt.ylabel("Radiative forcing [W m$^{-2}$]")
plt.title("Idealized 1%/yr CO2 forcing")
plt.grid(True)
plt.show()

plt.figure()
plt.plot(time, T)
plt.axvline(70, linestyle="--", label="CO2 doubling")
plt.xlabel("Time [years]")
plt.ylabel("Global mean temperature anomaly [K]")
plt.title("EBM response to 1%/yr CO2 increase")
plt.legend()
plt.grid(True)
plt.show()

## 13. TCR と ECS の違い / Difference between TCR and ECS

**ECS** は、CO2 倍増状態が十分長く続いた後の平衡温度上昇です。  
**ECS** is the equilibrium warming after a CO2 doubling is maintained for a sufficiently long time.

**TCR** は、CO2 が 1 %/yr で増加し、ちょうど倍増した時点の温度上昇です。  
**TCR** is the warming at the time of CO2 doubling in a 1%/yr CO2 increase experiment.

```text
ECS: 平衡応答
TCR: 過渡応答
```

```text
ECS: equilibrium response
TCR: transient response
```

海洋の熱吸収による遅れがあるため、通常 TCR は ECS より小さくなります。  
Because of delayed warming due to ocean heat uptake, TCR is usually smaller than ECS.

In [ ]:
F2x = 3.7
lam = 1.2

ECS = F2x / lam
TCR = T[np.argmin(np.abs(time - 70))]

print(f"ECS = {ECS:.2f} K")
print(f"TCR = {TCR:.2f} K")
print(f"TCR / ECS = {TCR / ECS:.2f}")

## 14. Mini CMIP: 複数の仮想モデルを比較する / Mini CMIP: comparing multiple idealized models

実際の CMIP では、多くの気候モデルを比較します。  
In actual CMIP, many climate models are compared.

ここでは、\(\lambda\) と \(C\) が異なる 5 つの仮想モデルを作ります。  
Here, we create five idealized models with different \(\lambda\) and \(C\).

それぞれについて、ECS と TCR を計算します。  
For each model, we compute ECS and TCR.

In [ ]:
models = pd.DataFrame({
    "model": ["Model A", "Model B", "Model C", "Model D", "Model E"],
    "lambda": [0.8, 1.0, 1.2, 1.5, 1.8],
    "C": [8, 15, 10, 30, 50]
})

models

In [ ]:
results = []

plt.figure()

for _, row in models.iterrows():
    model = row["model"]
    lam = row["lambda"]
    C = row["C"]

    time, T, F = run_ebm_timevarying(forcing_1pctCO2, lam=lam, C=C, years=150)
    ecs = 3.7 / lam
    tcr = T[np.argmin(np.abs(time - 70))]

    results.append({
        "model": model,
        "lambda": lam,
        "C": C,
        "ECS [K]": ecs,
        "TCR [K]": tcr,
        "TCR/ECS": tcr / ecs,
    })

    plt.plot(time, T, label=model)

plt.axvline(70, linestyle="--", label="CO2 doubling")
plt.xlabel("Time [years]")
plt.ylabel("Global mean temperature anomaly [K]")
plt.title("Mini CMIP: different model responses")
plt.legend()
plt.grid(True)
plt.show()

df_results = pd.DataFrame(results)
df_results

### 問い 3 / Question 3

ECS が大きいモデルは、必ず TCR も大きいでしょうか。  
Does a model with larger ECS always have larger TCR?

熱容量 \(C\) が大きいモデルでは、TCR はどうなりやすいでしょうか。  
What tends to happen to TCR when heat capacity \(C\) is large?

ECS は主に \(\lambda\) で決まります。  
ECS is mainly determined by \(\lambda\).

TCR は \(\lambda\) だけでなく、熱容量 \(C\) にも強く影響されます。  
TCR is affected not only by \(\lambda\), but also strongly by heat capacity \(C\).

## 15. 自由実験 / Your own experiment

ここは自由に値を変えるセルです。  
This is a free experiment cell.

以下を変えてみてください。  
Try changing:

```python
my_lambda
my_C
my_years
```

実行する前に、ECS と TCR がどうなるか予想してください。  
Before running, predict how ECS and TCR will change.

In [ ]:
my_lambda = 1.0
my_C = 20.0
my_years = 150

time, T, F = run_ebm_timevarying(
    forcing_1pctCO2,
    lam=my_lambda,
    C=my_C,
    years=my_years
)

my_ECS = 3.7 / my_lambda
my_TCR = T[np.argmin(np.abs(time - 70))]

plt.figure()
plt.plot(time, T)
plt.axvline(70, linestyle="--", label="CO2 doubling")
plt.xlabel("Time [years]")
plt.ylabel("Global mean temperature anomaly [K]")
plt.title("Your own simple climate model")
plt.legend()
plt.grid(True)
plt.show()

print(f"lambda = {my_lambda}")
print(f"C = {my_C}")
print(f"ECS = {my_ECS:.2f} K")
print(f"TCR = {my_TCR:.2f} K")
print(f"TCR/ECS = {my_TCR / my_ECS:.2f}")

## 16. 後続の海洋ボックスモデルとのつながり / Connection to later ocean box models

ここまでで学んだ構造は、海洋ボックスモデルでも同じです。  
The structure learned here is the same in the later ocean box models.

EBM では、変数は \(T\) だけでした。  
In the EBM, the only variable was \(T\).

海洋ボックスモデルでは、変数が増えます。  
In ocean box models, the number of variables increases.

```text
T
```

から、

```text
PO4S, PO4D
DICS, DICD
ALKS, ALKD
O2S, O2D
pCO2A
```

へ進みます。  
We move from \(T\) to tracers such as PO4, DIC, ALK, O2, and atmospheric pCO2.

しかし、プログラムの構造は同じです。  
However, the programming structure is the same.

```text
現在の値
↓
変化量を計算
↓
次の値を計算
↓
更新
↓
繰り返す
```

```text
current value
↓
compute tendency
↓
compute next value
↓
update
↓
repeat
```

## 17. たくさんの変数をまとめる / Putting many variables together

海洋ボックスモデルでは、変数が多くなります。  
In ocean box models, there are many variables.

例えば 2-box モデルでは、

```python
PO4S, PO4D
DICS, DICD
ALKS, ALKD
DO2S, DO2D
```

を同時に扱います。  
For example, in the 2-box model we handle PO4, DIC, ALK, and O2 in both surface and deep boxes.

これらを関数に渡すとき、1 つずつ渡すと大変です。  
Passing all of them one by one to a function is inconvenient.

そこで、辞書にまとめます。  
So we put them into a dictionary.

これは特別な概念ではありません。  
This is not a special concept.

単に「たくさんの値を入れる袋」です。  
It is simply a bag that stores many values.

In [ ]:
ocean = {
    "PO4S": 2.2,
    "PO4D": 2.2,
    "DICS": 2258.0,
    "DICD": 2258.0,
    "ALKS": 2371.0,
    "ALKD": 2371.0,
    "O2S": 170.0,
    "O2D": 170.0,
}

ocean

## 18. 辞書の値を更新する / Updating values in a dictionary

辞書に入っていても、やっていることは同じです。  
Even if values are stored in a dictionary, the idea is the same.

```python
ocean["PO4S"] = ocean["PO4S"] + change
```

これは、

```python
PO4S = PO4S + change
```

と同じ考え方です。  
This is the same idea as `PO4S = PO4S + change`.

2-box Notebook では、この考え方を使って `one_step_two_box()` を作ります。  
In the 2-box notebook, we use this idea to build `one_step_two_box()`.

In [ ]:
print("Before:", ocean["PO4S"])

change = -0.01
ocean["PO4S"] = ocean["PO4S"] + change

print("After :", ocean["PO4S"])

## 19. まとめ / Summary

この Notebook で学んだことは、次の通りです。  
What we learned in this notebook:

1. モデルは現実を単純化したものである。  
   A model is a simplification of reality.

2. 0D EBM は、放射強制力、フィードバック、熱容量だけで温度応答を表す。  
   A 0D EBM represents temperature response using forcing, feedback, and heat capacity.

3. 微分方程式は、時間刻みを使って次の値を計算できる。  
   A differential equation can be advanced by computing the next value using a time step.

4. `for` 文は時間積分である。  
   A for-loop is time integration.

5. ECS は平衡応答、TCR は過渡応答である。  
   ECS is an equilibrium response, while TCR is a transient response.

6. 熱容量が大きいと、応答は遅くなる。  
   Larger heat capacity slows the response.

7. 辞書は、たくさんの変数をまとめるための入れ物である。  
   A dictionary is a container for many variables.

次の Notebook では、海洋を 1 つの箱として扱い、PO4, DIC, O2 の保存則を書きます。  
In the next notebook, we treat the ocean as one box and write conservation equations for PO4, DIC, and O2.

## 20. レポート課題例 / Example report questions

### 課題 1 / Exercise 1

\(\lambda\) を 0.7, 1.0, 1.5, 2.0 に変えたとき、ECS はどう変わるか。  
How does ECS change when \(\lambda\) is changed to 0.7, 1.0, 1.5, and 2.0?

### 課題 2 / Exercise 2

\(C\) を変えたとき、平衡温度と応答速度はどう変わるか。  
How do equilibrium temperature and response speed change when \(C\) is changed?

### 課題 3 / Exercise 3

1 %/yr CO2 増加実験で、TCR が ECS より小さくなる理由を説明せよ。  
Explain why TCR is smaller than ECS in a 1%/yr CO2 increase experiment.

### 課題 4 / Exercise 4

Mini CMIP の仮想モデルのうち、ECS と TCR の違いが大きいモデルを選び、その理由を説明せよ。  
Choose a model in the Mini CMIP experiment with a large difference between ECS and TCR, and explain why.

### 課題 5 / Exercise 5

EBM と海洋ボックスモデルに共通するプログラム構造を説明せよ。  
Explain the common programming structure shared by the EBM and ocean box models.